1: Import Required Libraries

In [0]:
from pyspark.sql.functions import (
    col,
    lower,
    trim,
    regexp_replace,
    concat_ws,
    size,
    split
)

from pyspark.ml.feature import (
    RegexTokenizer,
    StopWordsRemover
)

2: Project Configuration

In [0]:
VALIDATED_PATH = "/Volumes/bda_project/projectdata/gold/validated_reviews/"

PREPROCESSED_PATH = "/Volumes/bda_project/projectdata/gold/preprocessed_reviews/"

3: Read Validated Dataset

In [0]:
df = spark.read.parquet(VALIDATED_PATH)

5: Dataset Overview

In [0]:
total_rows = df.count()

total_columns = len(df.columns)

print("=" * 50)

print(f"Rows         : {total_rows:,}")
print(f"Columns      : {total_columns}")
print("=" * 50)

Rows         : 2,983,784
Columns      : 23


6: Display Schema

In [0]:
df.printSchema()

root
 |-- parent_asin: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- review_rating: double (nullable = true)
 |-- review_title: string (nullable = true)
 |-- review_text: string (nullable = true)
 |-- helpful_vote: integer (nullable = true)
 |-- verified_purchase: boolean (nullable = true)
 |-- review_timestamp: timestamp (nullable = true)
 |-- review_date: date (nullable = true)
 |-- review_year: integer (nullable = true)
 |-- review_month: integer (nullable = true)
 |-- product_title: string (nullable = true)
 |-- store: string (nullable = true)
 |-- main_category: string (nullable = true)
 |-- sub_category: string (nullable = true)
 |-- product_average_rating: double (nullable = true)
 |-- product_rating_count: long (nullable = true)
 |-- description_text: string (nullable = true)
 |-- features_text: string (nullable = true)
 |-- product_image_url: string (nullable = true)
 |-- character_count: integer (nullable = true)
 |-- word_count: integer (nullable = tru

7: Display Sample Records

8: Verify Required Columns

In [0]:
required_columns = [
    "product_title",
    "review_title",
    "review_text",
    "description_text",
    "features_text"
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise Exception(f"Missing Columns : {missing_columns}")

print("All required columns are available.")

All required columns are available.


9: Combine Text Columns

In [0]:
from pyspark.sql.functions import concat_ws, col

df = df.withColumn(
    "combined_text",
    concat_ws(
        " ",
        col("product_title"),
        col("review_title"),
        col("review_text"),
        col("description_text"),
        col("features_text")
    )
)

10: Convert to Lowercase

In [0]:
from pyspark.sql.functions import lower

df = df.withColumn(
    "combined_text",
    lower(col("combined_text"))
)

11: Remove HTML Tags

In [0]:
from pyspark.sql.functions import regexp_replace

df = df.withColumn(
    "combined_text",
    regexp_replace(
        col("combined_text"),
        "<[^>]+>",
        ""
    )
)

12: Remove URLs

In [0]:
df = df.withColumn(
    "combined_text",
    regexp_replace(
        col("combined_text"),
        r"https?://\\S+|www\\.\\S+",
        ""
    )
)

13: Remove Email Addresses

In [0]:
df = df.withColumn(
    "combined_text",
    regexp_replace(
        col("combined_text"),
        r"\\S+@\\S+",
        ""
    )
)

14: Remove Emojis

In [0]:
emoji_pattern = (
    "["
    "\U0001F600-\U0001F64F"
    "\U0001F300-\U0001F5FF"
    "\U0001F680-\U0001F6FF"
    "\U0001F1E0-\U0001F1FF"
    "\U00002700-\U000027BF"
    "\U000024C2-\U0001F251"
    "]+"
)

df = df.withColumn(
    "combined_text",
    regexp_replace(
        col("combined_text"),
        emoji_pattern,
        ""
    )
)

15: Expand Contractions

In [0]:
# =====================================
# Expand Contractions
# =====================================

contractions = {
    "can't": "cannot",
    "won't": "will not",
    "don't": "do not",
    "didn't": "did not",
    "doesn't": "does not",
    "isn't": "is not",
    "aren't": "are not",
    "wasn't": "was not",
    "weren't": "were not",
    "haven't": "have not",
    "hasn't": "has not",
    "hadn't": "had not",
    "shouldn't": "should not",
    "wouldn't": "would not",
    "couldn't": "could not",
    "mustn't": "must not",
    "it's": "it is",
    "i'm": "i am",
    "you're": "you are",
    "they're": "they are",
    "we're": "we are",
    "that's": "that is",
    "there's": "there is",
    "what's": "what is",
    "who's": "who is",
    "let's": "let us"
}

In [0]:
for contraction, expanded in contractions.items():

    df = df.withColumn(
        "combined_text",
        regexp_replace(
            col("combined_text"),
            rf"\b{contraction}\b",
            expanded
        )
    )

16: Remove Numbers

In [0]:
df = df.withColumn(
    "combined_text",
    regexp_replace(
        col("combined_text"),
        "[0-9]+",
        ""
    )
)

17: Remove Punctuation

In [0]:
df = df.withColumn(
    "combined_text",
    regexp_replace(
        col("combined_text"),
        "[^a-zA-Z\\s]",
        " "
    )
)

18: Normalize Spaces

In [0]:
df = df.withColumn(
    "combined_text",
    regexp_replace(
        col("combined_text"),
        "\\s+",
        " "
    )
)

df = df.withColumn(
    "combined_text",
    trim(col("combined_text"))
)

19: Tokenization

In [0]:
tokenizer = RegexTokenizer(
    inputCol="combined_text",
    outputCol="tokens",
    pattern="\\W+"
)

df = tokenizer.transform(df)

In [0]:

remover = StopWordsRemover(
    inputCol="tokens",
    outputCol="filtered_tokens"
)

df = remover.transform(df)